# 04 — Conservation Laws: Energy, Momentum & Angular Momentum

Conservation laws are the backbone of gravitational dynamics. In your black hole research, the **same conservation laws** appear as the Carter constant, conserved energy E, and angular momentum L_z along null geodesics.

## What You Will Do
1. Verify energy conservation numerically for all integrators
2. Visualize KE ↔ PE exchange (virial theorem)
3. Prove angular momentum conservation
4. Understand why symplectic integrators are superior for long runs
5. Connect to Killing vectors and geodesic conserved quantities in GR

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt

from src.core.initial_conditions import figure_eight, lagrange_triangle, build_state_vector
from src.core.integrator import integrate_scipy, integrate_leapfrog, integrate_rk4
from src.core.equations import (
    transform_to_cm_frame, total_energy, total_angular_momentum, G_UNITS
)
from src.analysis.analysis import angular_momentum_error, kinetic_energy_partition
from src.visualization.visualize import BODY_COLORS

print(f'G = {G_UNITS:.6f} AU³/(M☉·yr²) = 4π² (exact in geometric units)')

## 1. The Three Conservation Laws

For an isolated gravitational N-body system:

**Energy:** $E = \underbrace{\frac{1}{2}\sum_i m_i |v_i|^2}_{\text{KE}} - \underbrace{G\sum_{i<j}\frac{m_i m_j}{|r_i - r_j|}}_{-\text{PE}} = \text{const}$

**Linear momentum:** $\mathbf{P} = \sum_i m_i \mathbf{v}_i = \text{const}$ (Newton's 3rd law)

**Angular momentum:** $L = \sum_i m_i (\mathbf{r}_i \times \mathbf{v}_i) = \text{const}$ (rotational symmetry)

**GR connection:** In Schwarzschild/Kerr spacetime, these become:
- $E = -p_t$ (conserved from time-translation Killing vector $\partial_t$)
- $L_z = p_\phi$ (conserved from azimuthal Killing vector $\partial_\phi$)
- Carter constant $Q$ (from hidden symmetry)

The physical idea is the same — symmetry → conserved quantity (Noether's theorem).

In [ ]:
# Setup: figure-eight in CM frame
ic = figure_eight()
masses = np.array([ic['m1'], ic['m2'], ic['m3']])
r1,r2,r3,v1,v2,v3 = transform_to_cm_frame(
    ic['r1'],ic['r2'],ic['r3'],
    ic['v1'],ic['v2'],ic['v3'], *masses)
ic.update({'r1':r1,'r2':r2,'r3':r3,'v1':v1,'v2':v2,'v3':v3})
y0 = build_state_vector(ic)

t_end  = 30.0
t_eval = np.linspace(0, t_end, 10000)

result = integrate_scipy(y0, masses, (0, t_end), t_eval, method='DOP853', rtol=1e-12, atol=1e-14)
print(f'Steps: {result.n_steps}  |  Max ΔE/E₀: {result.max_energy_error:.2e}')

## 2. Energy: KE ↔ PE Exchange (Virial Theorem)

In [ ]:
pos = result.positions; vel = result.velocities

# Compute KE and PE at each timestep
KE = np.array([
    0.5 * sum(masses[b] * np.dot(vel[i,b], vel[i,b]) for b in range(3))
    for i in range(result.n_steps)
])
PE = result.energy - KE

fig, axes = plt.subplots(2,2,figsize=(13,8))

# KE and PE vs time
axes[0,0].plot(result.t, KE, color='#E8632A', lw=1, label='Kinetic Energy')
axes[0,0].plot(result.t, PE, color='#2A7AE8', lw=1, label='Potential Energy')
axes[0,0].plot(result.t, result.energy, color='#2AE87A', lw=1.5, ls='--', label='Total Energy')
axes[0,0].set_xlabel('Time [yr]'); axes[0,0].set_ylabel('Energy [M☉·AU²/yr²]')
axes[0,0].set_title('Energy Components')
axes[0,0].legend(fontsize=8); axes[0,0].grid(True,alpha=0.2)

# Virial ratio <KE>/<PE> = -1/2 for bound systems (virial theorem)
virial = KE / (-PE + 1e-30)
axes[0,1].plot(result.t, virial, color='#E8C32A', lw=0.8)
axes[0,1].axhline(0.5, color='#2AE87A', ls='--', lw=1, label='Virial: KE/|PE| = 0.5')
axes[0,1].set_xlabel('Time [yr]'); axes[0,1].set_ylabel('KE / |PE|')
axes[0,1].set_title('Virial Ratio (time-average → 0.5 for bound systems)')
axes[0,1].legend(fontsize=8); axes[0,1].grid(True,alpha=0.2)
axes[0,1].set_ylim(0, 2)

# Energy conservation error
axes[1,0].semilogy(result.t, result.energy_error + 1e-20, color='#E8632A', lw=1)
axes[1,0].axhline(1e-8, color='#2AE87A', ls='--', lw=0.8, label='1e-8 threshold')
axes[1,0].set_xlabel('Time [yr]'); axes[1,0].set_ylabel('|ΔE/E₀|')
axes[1,0].set_title('Relative Energy Error (DOP853)')
axes[1,0].legend(fontsize=8); axes[1,0].grid(True,alpha=0.2)

# KE partition between bodies
ke_frac = kinetic_energy_partition(result)
for b,c in enumerate(BODY_COLORS):
    axes[1,1].plot(result.t, ke_frac[:,b], color=c, lw=0.8, label=f'Body {b+1}')
axes[1,1].set_xlabel('Time [yr]'); axes[1,1].set_ylabel('KE fraction')
axes[1,1].set_title('Kinetic Energy Partition (equal for choreography)')
axes[1,1].legend(fontsize=8); axes[1,1].grid(True,alpha=0.2)

plt.suptitle('Conservation Laws — Figure-Eight Choreography', fontsize=12)
plt.tight_layout(); plt.show()

## 3. Angular Momentum Conservation

In [ ]:
from src.analysis.analysis import compute_specific_angular_momentum

fig, axes = plt.subplots(1,2,figsize=(12,5))

# Total angular momentum
L0 = result.angular_momentum[0]
L_err = angular_momentum_error(result)

axes[0].plot(result.t, result.angular_momentum, color='#2A7AE8', lw=1.2)
axes[0].axhline(L0, color='#2AE87A', ls='--', lw=0.8, label=f'L₀ = {L0:.6f}')
axes[0].set_xlabel('Time [yr]'); axes[0].set_ylabel('L [M☉·AU²/yr]')
axes[0].set_title('Total Angular Momentum (should be constant)')
axes[0].legend(fontsize=9); axes[0].grid(True,alpha=0.2)

# Error
axes[1].semilogy(result.t, L_err + 1e-20, color='#2A7AE8', lw=1)
axes[1].set_xlabel('Time [yr]'); axes[1].set_ylabel('|ΔL/L₀|')
axes[1].set_title('Angular Momentum Relative Error')
axes[1].grid(True,alpha=0.2)

plt.tight_layout(); plt.show()
print(f'Max |ΔL/L₀| = {L_err.max():.2e}  (should be < 1e-8 for DOP853)')

## 4. Symplectic vs. Non-Symplectic: Long-Run Energy Drift

In [ ]:
ic = lagrange_triangle()
masses = np.array([ic['m1'],ic['m2'],ic['m3']])
y0 = build_state_vector(ic)
t_end = 50.0

print('Running integrators (this takes ~30 seconds)...')
res_dop = integrate_scipy(y0, masses, (0,t_end),
                           np.linspace(0,t_end,15000), method='DOP853', rtol=1e-10)
res_lf  = integrate_leapfrog(y0, masses, (0,t_end), dt=0.002, store_every=5)
res_rk4 = integrate_rk4(y0,     masses, (0,t_end), dt=0.002, store_every=5)
print('Done.')

fig, ax = plt.subplots(figsize=(11,5))
ax.semilogy(res_dop.t, res_dop.energy_error+1e-20, color='#2AE87A', lw=1.2, label=f'DOP853  (max={res_dop.max_energy_error:.1e})')
ax.semilogy(res_lf.t,  res_lf.energy_error +1e-20, color='#2A7AE8', lw=1.0, label=f'Leapfrog symplectic (max={res_lf.max_energy_error:.1e})')
ax.semilogy(res_rk4.t, res_rk4.energy_error+1e-20, color='#E8632A', lw=1.0, ls='--', label=f'RK4 fixed-step (max={res_rk4.max_energy_error:.1e})')

ax.set_xlabel('Time [yr]',fontsize=12); ax.set_ylabel('|ΔE/E₀|',fontsize=12)
ax.set_title('Energy Conservation: Symplectic vs. Non-Symplectic (Lagrange Triangle, 50 yr)', fontsize=12)
ax.legend(fontsize=9); ax.grid(True,alpha=0.25)

ax.text(0.02,0.05,
        'Leapfrog oscillates but does NOT drift\n(conserves modified Hamiltonian H̃)\nRK4 shows linear secular drift over time',
        transform=ax.transAxes,va='bottom',fontsize=8,color='#8B949E',
        bbox=dict(boxstyle='round',facecolor='#21262D',alpha=0.7))

plt.tight_layout(); plt.show()

## 5. Connection to GR: Conservation Laws from Symmetry

The Newtonian conservation laws you verified here have direct GR analogs:

| Newtonian | GR equivalent | Killing vector |
|-----------|---------------|----------------|
| Energy E = const | $E = -p_\mu \xi^\mu$ where $\xi^\mu = (1,0,0,0)$ | $\partial_t$ (time symmetry) |
| $L_z$ = const | $L_z = p_\mu \psi^\mu$ where $\psi^\mu = (0,0,0,1)$ | $\partial_\phi$ (azimuthal symmetry) |
| No Newtonian analog | Carter constant $Q$ | Hidden symmetry of Kerr |

Your ray tracer enforces these conservation laws at every integration step — monitoring their drift is how you validate your Kerr geodesic integrator.

In [ ]:
# Demonstrate: in CM frame, total momentum is exactly zero
r1,r2,r3,v1,v2,v3 = transform_to_cm_frame(
    ic['r1'],ic['r2'],ic['r3'],
    ic['v1'],ic['v2'],ic['v3'], *masses)

p_total_initial = masses[0]*v1 + masses[1]*v2 + masses[2]*v3
print(f'Total momentum in CM frame:')
print(f'  Px = {p_total_initial[0]:.2e}  (should be 0)')
print(f'  Py = {p_total_initial[1]:.2e}  (should be 0)')
print()
print(f'This is the Newtonian analog of Σ m_i v_i = 0,')
print(f'which in GR becomes: the center-of-mass frame condition.')
print(f'In your ray tracer: the observer is in the CM frame of the BH.')

## ✅ Summary

- Energy is conserved to ~1e-10 with DOP853, oscillates (not drifts) with Leapfrog, drifts with RK4
- Angular momentum conservation confirms correct force calculation
- KE and PE exchange continuously; time-averaged ratio → 0.5 (virial theorem)
- Symplectic integrators (Leapfrog) preserve a modified Hamiltonian → no secular drift
- Newtonian conservation laws ↔ GR Killing vectors — same physics, different language

**Next:** `05_research_connection.ipynb` — bridge from three-body physics to Kerr geodesics and your paper.